# GS 2026.1 — Computação Quântica e IA
## Quantum Machine Learning para Detecção de Anomalias em Telemetria de Satélites

**FIAP | Turma 2TIAP | Global Solution 2026.1**  
**Domínio:** Manutenção Espacial — Detecção de Anomalias em Satélites LEO  
**ODS:** 9 · 11 · 13  
**`random_state = 42`** — notebook 100% reproduzível

---

### Objetivo
Aplicar **QSVC** (Quantum Support Vector Classifier) com `ZZFeatureMap` em dados de telemetria de satélite para **classificar anomalias operacionais**, comparando com baselines clássicos (SVM-RBF e Random Forest).

### Stack
- Python 3.10+, Qiskit 1.2.4, qiskit-machine-learning 0.7.2, qiskit-aer 0.14.2
- scikit-learn, imbalanced-learn, pandas, numpy, matplotlib, seaborn


In [ ]:
# ============================================================
# CÉLULA 1 — Instalação (descomente para Google Colab)
# ============================================================
# !pip install qiskit==1.2.4 qiskit-machine-learning==0.7.2 qiskit-aer==0.14.2 \
#             qiskit-algorithms==0.3.1 scikit-learn imbalanced-learn \
#             pandas numpy matplotlib seaborn scipy


In [ ]:
# ============================================================
# CÉLULA 2 — Imports e Configuração Global
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from collections import Counter

# Scikit-learn
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, roc_curve, classification_report
)
from imblearn.over_sampling import SMOTE

# Qiskit 1.x
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

# Seed global — OBRIGATÓRIO para reproducibilidade
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Bibliotecas carregadas com sucesso!")
print(f"random_state = {RANDOM_STATE} (fixado em todas as operacoes aleatorias)")

import qiskit
import qiskit_machine_learning
print(f"Qiskit versao              : {qiskit.__version__}")
print(f"qiskit-machine-learning    : {qiskit_machine_learning.__version__}")


---
## 1. Dataset Espacial e Pré-processamento

### Fonte dos dados
Dataset sintético inspirado no **NASA MSAD** (Mission Sensor Anomaly Dataset) e nos padrões de telemetria de satélites LEO (Low Earth Orbit). As distribuições dos parâmetros físicos (temperatura, tensão, corrente, giroscópio) seguem as especificações de satélites de órbita baixa documentadas pela NASA e ESA.

- **Referência:** NASA MSAD — https://nasa.github.io/MSAD/
- **Domínio:** Manutenção Espacial — detecção de falhas antes de impacto na missão
- **Problema:** Classificação binária — `0 = Operação Normal` | `1 = Anomalia`


In [ ]:
# ============================================================
# CÉLULA 3 — Geração do Dataset (NASA MSAD-style)
# ============================================================
def gerar_telemetria_satelite(n_total=600, proporcao_anomalia=0.30, seed=42):
    """
    Gera dataset sintetico de telemetria de satelite LEO.

    Parametros baseados em especificacoes tipicas de satelites de orbita baixa.
    Fonte de referencia: NASA MSAD, ESA Sentinel-6 telemetry specs.

    Features (8 sensores):
        temperatura  : temperatura interna do compartimento [degC]
        pressao      : pressao interna [kPa]
        radiacao     : nivel de radiacao ionizante [mSv/h]
        tensao       : tensao do barramento de energia [V] (padrao 28V)
        corrente     : corrente consumida [A]
        gyro_x/y/z   : velocidade angular dos 3 eixos [deg/s] (IMU)

    Classes:
        0 = Normal (operacao dentro dos limites nominais)
        1 = Anomalia (falha termica, pico de radiacao ou instabilidade de atitude)
    """
    rng = np.random.RandomState(seed)
    n_anomalia = int(n_total * proporcao_anomalia)
    n_normal = n_total - n_anomalia

    # --- Operacao Normal ---
    temperatura_n = rng.normal(22.0, 4.5, n_normal)
    pressao_n     = rng.normal(101.5, 1.8, n_normal)
    radiacao_n    = rng.normal(0.28, 0.045, n_normal)
    tensao_n      = rng.normal(28.2, 0.42, n_normal)
    corrente_n    = rng.normal(5.15, 0.28, n_normal)
    gyro_x_n      = rng.normal(0.0, 0.08, n_normal)
    gyro_y_n      = rng.normal(0.0, 0.08, n_normal)
    gyro_z_n      = rng.normal(0.0, 0.08, n_normal)

    # --- Anomalias ---
    # Simula: falha termica, pico de radiacao, instabilidade de atitude e
    # queda de tensao (typical failure modes em satelites LEO)
    temperatura_a = rng.normal(52.0, 9.5, n_anomalia)
    pressao_a     = rng.normal(93.0, 4.5, n_anomalia)
    radiacao_a    = rng.normal(0.85, 0.18, n_anomalia)
    tensao_a      = rng.normal(23.5, 2.2, n_anomalia)
    corrente_a    = rng.normal(7.8, 0.95, n_anomalia)
    gyro_x_a      = rng.normal(0.55, 0.32, n_anomalia)
    gyro_y_a      = rng.normal(-0.42, 0.28, n_anomalia)
    gyro_z_a      = rng.normal(0.62, 0.22, n_anomalia)

    df_n = pd.DataFrame({
        'temperatura': temperatura_n, 'pressao': pressao_n,
        'radiacao': radiacao_n,       'tensao': tensao_n,
        'corrente': corrente_n,       'gyro_x': gyro_x_n,
        'gyro_y': gyro_y_n,           'gyro_z': gyro_z_n,
        'anomalia': 0
    })
    df_a = pd.DataFrame({
        'temperatura': temperatura_a, 'pressao': pressao_a,
        'radiacao': radiacao_a,       'tensao': tensao_a,
        'corrente': corrente_a,       'gyro_x': gyro_x_a,
        'gyro_y': gyro_y_a,           'gyro_z': gyro_z_a,
        'anomalia': 1
    })
    df = pd.concat([df_n, df_a], ignore_index=True)
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    return df


df = gerar_telemetria_satelite(n_total=600, seed=RANDOM_STATE)

print("=" * 58)
print("  DATASET: Telemetria de Satelite LEO (NASA MSAD-style)")
print("=" * 58)
print(f"  Total de registros   : {len(df)}")
print(f"  Features             : {df.shape[1] - 1} sensores")
print(f"  Periodo simulado     : 2024-01-01 a 2024-12-31 (horario)")
print(f"  Fonte de referencia  : NASA MSAD / ESA Sentinel specs")
print()
print("  Distribuicao de classes:")
vc = df['anomalia'].value_counts()
for lbl, cnt in vc.sort_index().items():
    nome = 'Normal  ' if lbl == 0 else 'Anomalia'
    print(f"    Classe {lbl} ({nome}): {cnt} registros ({cnt/len(df)*100:.1f}%)")
print()
print(df.describe().round(3))


In [ ]:
# ============================================================
# CÉLULA 4 — Dicionário de Dados
# ============================================================
dicionario = pd.DataFrame({
    'Feature'     : ['temperatura','pressao','radiacao','tensao','corrente',
                     'gyro_x','gyro_y','gyro_z','anomalia'],
    'Descricao'   : [
        'Temperatura interna do compartimento eletronico',
        'Pressao interna do modulo de servico',
        'Nivel de radiacao ionizante detectada pelo dosimetro',
        'Tensao do barramento de energia principal (padrao 28V)',
        'Corrente eletrica total consumida pelo satelite',
        'Velocidade angular - eixo X (IMU)',
        'Velocidade angular - eixo Y (IMU)',
        'Velocidade angular - eixo Z (IMU)',
        'Rotulo binario de anomalia (0=Normal, 1=Anomalia)'
    ],
    'Unidade'     : ['degC','kPa','mSv/h','V','A','deg/s','deg/s','deg/s','binario'],
    'Faixa Normal': [
        '15 a 30','-20 a 60 (cabin)','0.18 a 0.38','27 a 29','4.6 a 5.7',
        '+/- 0.25','+/- 0.25','+/- 0.25','0 ou 1'
    ],
    'Sensor'      : ['TMP117','BMP388','RAD3B','INA226','INA226',
                     'ICM-42688','ICM-42688','ICM-42688','Ground Truth']
})
print("DICIONARIO DE DADOS")
print("=" * 100)
print(dicionario.to_string(index=False))
print("=" * 100)
print("\nObservacoes:")
print("  - Pre-processamento: MinMaxScaler para [-pi, pi] + PCA (4 componentes = 4 qubits)")
print("  - Split treino/teste ANTES de qualquer transformacao (sem data leakage)")
print("  - Freq. de amostragem simulada: 1 leitura/hora | 600 registros totais")


In [ ]:
# ============================================================
# CÉLULA 5 — Análise Exploratória (EDA)
# ============================================================
features_plot = ['temperatura','pressao','radiacao','tensao',
                 'corrente','gyro_x','gyro_y','gyro_z']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('EDA — Distribuicao das Features por Classe\n'
             'Telemetria de Satelite LEO (NASA MSAD-style)',
             fontsize=13, fontweight='bold')

cores = {0: '#2196F3', 1: '#F44336'}
nomes = {0: 'Normal', 1: 'Anomalia'}

for i, feat in enumerate(features_plot):
    ax = axes[i // 4, i % 4]
    for lbl in [0, 1]:
        subset = df[df['anomalia'] == lbl][feat]
        ax.hist(subset, bins=30, alpha=0.6, color=cores[lbl],
                label=nomes[lbl], density=True)
    ax.set_title(feat, fontweight='bold')
    ax.legend(fontsize=8)
    ax.set_ylabel('Densidade', fontsize=8)

plt.tight_layout()
plt.savefig('eda_distribuicoes.png', dpi=150, bbox_inches='tight')
plt.show()
print("EDA salva em 'eda_distribuicoes.png'")

# Matriz de correlacao
fig, ax = plt.subplots(figsize=(9, 7))
corr = df[features_plot + ['anomalia']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, linewidths=0.5)
ax.set_title('Matriz de Correlacao — Telemetria de Satelite', fontweight='bold')
plt.tight_layout()
plt.savefig('eda_correlacao.png', dpi=150, bbox_inches='tight')
plt.show()
print("Correlacao salva em 'eda_correlacao.png'")


---
## Pré-processamento

Pipeline de pré-processamento para integração híbrida clássico-quântica:

1. **Train/test split** antes de qualquer transformação → evita data leakage
2. **MinMaxScaler** → range `[-π, π]` (ideal para encoding em rotações de qubits)
3. **PCA** → redução a 4 componentes = 4 qubits (variância explicada > 70%)
4. **SMOTE** → balanceamento de classes se necessário

> **Importante:** `fit` é aplicado APENAS nos dados de treino; `transform` nos dados de teste.


In [ ]:
# ============================================================
# CÉLULA 6 — Pré-processamento
# ============================================================
print("=" * 58)
print("  PRE-PROCESSAMENTO")
print("=" * 58)

FEATURES = ['temperatura','pressao','radiacao','tensao',
            'corrente','gyro_x','gyro_y','gyro_z']
N_QUBITS = 4  # 3 a 5 para NISQ; escolhido 4 para capturar 4 PCs

X = df[FEATURES].values
y = df['anomalia'].values

# 1) Split ANTES de qualquer transformacao (evitar data leakage)
X_train_raw, X_test_raw, y_train_raw, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f"  Split       : 80% treino / 20% teste (estratificado)")
print(f"  Train shape : {X_train_raw.shape}")
print(f"  Test shape  : {X_test_raw.shape}")

# 2) Normalizacao MinMaxScaler -> [-pi, pi]
scaler = MinMaxScaler(feature_range=(-np.pi, np.pi))
X_train_scaled = scaler.fit_transform(X_train_raw)   # fit apenas no train
X_test_scaled  = scaler.transform(X_test_raw)         # transform no test
print(f"\n  Normalizacao: MinMaxScaler -> [-pi, pi]")
print(f"  Range treino: [{X_train_scaled.min():.4f}, {X_train_scaled.max():.4f}]")
print(f"  Range teste : [{X_test_scaled.min():.4f}, {X_test_scaled.max():.4f}]")

# 3) PCA para N_QUBITS componentes
pca = PCA(n_components=N_QUBITS, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train_scaled)   # fit apenas no train
X_test_pca  = pca.transform(X_test_scaled)         # transform no test

var_exp = pca.explained_variance_ratio_.sum() * 100
print(f"\n  PCA         : {N_QUBITS} componentes (= numero de qubits)")
for i, v in enumerate(pca.explained_variance_ratio_):
    print(f"    PC{i+1}: {v*100:.1f}%")
print(f"  Variancia total explicada: {var_exp:.1f}%")
assert var_exp >= 60, f"Variancia explicada abaixo de 60%: {var_exp:.1f}%"
print(f"  OK — variancia >= 60% (ideal >= 70%)")

# 4) Verificar desbalanceamento
dist_train = Counter(y_train_raw)
razao = dist_train[1] / dist_train[0]
print(f"\n  Distribuicao treino: {dict(dist_train)}")
print(f"  Razao anomalia/normal: {razao:.3f}")

if razao < 0.40:
    smote = SMOTE(random_state=RANDOM_STATE)
    X_train_pca, y_train = smote.fit_resample(X_train_pca, y_train_raw)
    print(f"  SMOTE aplicado. Nova distribuicao: {dict(Counter(y_train))}")
else:
    y_train = y_train_raw
    print(f"  SMOTE nao necessario (razao {razao:.2f} >= 0.40)")

print("\n  Pre-processamento concluido (sem data leakage)")
print(f"  Train final : {X_train_pca.shape} | Test: {X_test_pca.shape}")


---
## 2. Implementação QML com Qiskit 1.x

### Modelo: QSVC com FidelityQuantumKernel + ZZFeatureMap

- **Feature Map:** `ZZFeatureMap` (reps=1) — captura correlações de segunda ordem entre as features, ideal para dados de sensores correlacionados (temperatura ↔ tensão, radiação ↔ corrente)
- **Nº de qubits:** 4 (= nº de componentes PCA)
- **Simulação:** Qiskit Aer `statevector_simulator` ou `qasm_simulator`
- **Subconjunto:** ≤ 300 amostras para treino quântico (limitação NISQ — kernel matrix O(n²))


In [ ]:
# ============================================================
# CÉLULA 7 — Subconjunto Quântico (limitacao NISQ)
# ============================================================
N_QUANTUM_TRAIN = 280  # <= 400 por limitacao do simulador

rng_q = np.random.RandomState(RANDOM_STATE)
idx_q = rng_q.choice(len(X_train_pca), size=N_QUANTUM_TRAIN, replace=False)

X_train_q = X_train_pca[idx_q]
y_train_q = y_train[idx_q]

print(f"Subconjunto quantico de treino: {N_QUANTUM_TRAIN} amostras")
print(f"Justificativa: kernel matrix tem custo O(n^2) * O(circuito)")
print(f"  n=280 -> ~78.400 avaliacoes de circuito (viavel)")
print(f"  n=480 -> ~230.400 avaliacoes (inviavel em tempo razoavel)")
print(f"Distribuicao quantum train: {dict(Counter(y_train_q))}")


In [ ]:
# ============================================================
# CÉLULA 8 — Feature Map e Circuito Quântico
# ============================================================
print("=" * 58)
print("  FEATURE MAP: ZZFeatureMap")
print("=" * 58)

feature_map = ZZFeatureMap(
    feature_dimension=N_QUBITS,
    reps=1,
    entanglement='linear'
)

print(f"  Tipo           : ZZFeatureMap")
print(f"  Qubits         : {N_QUBITS}")
print(f"  Reps           : 1")
print(f"  Entanglement   : linear")
print(f"  Params treinaveis: {len(feature_map.parameters)}")

# Decomposicao para medir profundidade
qc_decomp = feature_map.decompose()
print(f"  Profundidade   : {qc_decomp.depth()}")
print(f"  Contagem ops   : {dict(qc_decomp.count_ops())}")

print("\nCircuito ZZFeatureMap (parametros simbolicos):")
print(feature_map.draw(output='text', fold=90))


In [ ]:
# ============================================================
# CÉLULA 9 — QSVC: Treinamento e Inferência
# ============================================================
print("=" * 58)
print("  QSVC — Quantum Support Vector Classifier")
print("=" * 58)

# Configuracao do kernel quantico
# Tenta usar Qiskit Aer (qasm_simulator); fallback para statevector
backend_info = "statevector_simulator (padrao)"
try:
    from qiskit_aer.primitives import Sampler as AerSampler
    try:
        from qiskit_algorithms.state_fidelities import ComputeUncompute
    except ImportError:
        from qiskit.algorithms.state_fidelities import ComputeUncompute
    aer_sampler = AerSampler()
    fidelidade   = ComputeUncompute(sampler=aer_sampler)
    kernel = FidelityQuantumKernel(feature_map=feature_map, fidelity=fidelidade)
    backend_info = "Qiskit Aer (qasm_simulator, shots=1024)"
    print(f"  Backend: {backend_info}")
except Exception as exc:
    kernel = FidelityQuantumKernel(feature_map=feature_map)
    print(f"  Backend: {backend_info} (fallback — {exc})")

qsvc = QSVC(quantum_kernel=kernel)

print(f"\n  Treinando QSVC com {N_QUANTUM_TRAIN} amostras...")
t0 = time.time()
qsvc.fit(X_train_q, y_train_q)
T_QSVC_TRAIN = time.time() - t0
print(f"  Tempo de treinamento : {T_QSVC_TRAIN:.2f} s")

# Inferencia
t0 = time.time()
y_pred_qsvc = qsvc.predict(X_test_pca)
T_QSVC_INF = (time.time() - t0) / len(X_test_pca) * 1000

# Decision function para AUC-ROC
df_score_qsvc = qsvc.decision_function(X_test_pca)
scaler_auc = MinMaxScaler()
y_prob_qsvc = scaler_auc.fit_transform(df_score_qsvc.reshape(-1, 1)).ravel()

print(f"  Tempo de inferencia  : {T_QSVC_INF:.3f} ms/amostra")
print(f"  Amostras de treino   : {N_QUANTUM_TRAIN}")
print(f"  Amostras de teste    : {len(X_test_pca)}")
print(f"\n  QSVC treinado com sucesso!")
print("\nClassification Report - QSVC:")
print(classification_report(y_test, y_pred_qsvc, target_names=['Normal','Anomalia']))


In [ ]:
# ============================================================
# CÉLULA 10 — Kernel Matrix (visualizacao)
# ============================================================
print("Calculando Quantum Kernel Matrix (50 amostras para visualizacao)...")

N_KM = 50
idx_km = np.random.RandomState(RANDOM_STATE).choice(len(X_train_q), N_KM, replace=False)
X_km = X_train_q[idx_km]

t0 = time.time()
K_matrix = kernel.evaluate(X_km)
t_km = time.time() - t0

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(K_matrix, cmap='viridis', aspect='auto', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Fidelidade Quantica')
ax.set_title(f'Quantum Kernel Matrix\nZZFeatureMap ({N_QUBITS} qubits, reps=1) | {N_KM} amostras',
             fontweight='bold')
ax.set_xlabel('Amostra i')
ax.set_ylabel('Amostra j')
plt.tight_layout()
plt.savefig('kernel_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Kernel Matrix computada em {t_km:.2f}s | Shape: {K_matrix.shape}")
print(f"Diagonal (auto-fidelidade): min={np.diag(K_matrix).min():.4f} | max={np.diag(K_matrix).max():.4f}")


---
## 3. Baselines Clássicos

Treinados nos **mesmos dados PCA** que o QSVC para garantir comparação justa:
- **SVM-RBF:** `SVC(kernel='rbf', C=1.0, gamma='scale')`
- **Random Forest:** `RandomForestClassifier(n_estimators=100)`


In [ ]:
# ============================================================
# CÉLULA 11 — SVM-RBF e Random Forest
# ============================================================
print("=" * 58)
print("  BASELINES CLASSICOS")
print("=" * 58)

# Treinar com TODOS os dados de treino PCA (para comparacao justa
# os classicos podem usar mais dados; avaliacao e no mesmo test set)
X_tr_cl = X_train_pca
y_tr_cl = y_train

# --- SVM-RBF ---
print("\n[1] SVM com kernel RBF:")
svm_rbf = SVC(kernel='rbf', C=1.0, gamma='scale',
              probability=True, random_state=RANDOM_STATE)
t0 = time.time()
svm_rbf.fit(X_tr_cl, y_tr_cl)
T_SVM_TRAIN = time.time() - t0

t0 = time.time()
y_pred_svm = svm_rbf.predict(X_test_pca)
T_SVM_INF = (time.time() - t0) / len(X_test_pca) * 1000
y_prob_svm = svm_rbf.predict_proba(X_test_pca)[:, 1]

print(f"  Treino: {T_SVM_TRAIN:.4f}s | Inferencia: {T_SVM_INF:.4f} ms/amostra")
print(classification_report(y_test, y_pred_svm, target_names=['Normal','Anomalia']))

# --- Random Forest ---
print("[2] Random Forest:")
rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
t0 = time.time()
rf.fit(X_tr_cl, y_tr_cl)
T_RF_TRAIN = time.time() - t0

t0 = time.time()
y_pred_rf = rf.predict(X_test_pca)
T_RF_INF = (time.time() - t0) / len(X_test_pca) * 1000
y_prob_rf = rf.predict_proba(X_test_pca)[:, 1]

print(f"  Treino: {T_RF_TRAIN:.4f}s | Inferencia: {T_RF_INF:.4f} ms/amostra")
print(classification_report(y_test, y_pred_rf, target_names=['Normal','Anomalia']))


---
## 4. Benchmarking e Análise Comparativa


In [ ]:
# ============================================================
# CÉLULA 12 — Tabela de Métricas Comparativas
# ============================================================
def calcular_metricas(nome, y_true, y_pred, y_prob, t_treino, t_inf):
    return {
        'Modelo'        : nome,
        'Acuracia'      : round(accuracy_score(y_true, y_pred), 4),
        'F1-Score'      : round(f1_score(y_true, y_pred, average='weighted'), 4),
        'Precisao'      : round(precision_score(y_true, y_pred, average='weighted'), 4),
        'Recall'        : round(recall_score(y_true, y_pred, average='weighted'), 4),
        'AUC-ROC'       : round(roc_auc_score(y_true, y_prob), 4),
        'T.Treino(s)'   : round(t_treino, 3),
        'T.Inf(ms/am)'  : round(t_inf, 4)
    }

resultados = [
    calcular_metricas('QSVC (QML)',            y_test, y_pred_qsvc,
                      y_prob_qsvc,  T_QSVC_TRAIN, T_QSVC_INF),
    calcular_metricas('SVM-RBF (Classico)',    y_test, y_pred_svm,
                      y_prob_svm,   T_SVM_TRAIN,  T_SVM_INF),
    calcular_metricas('Random Forest (Class.)',y_test, y_pred_rf,
                      y_prob_rf,    T_RF_TRAIN,   T_RF_INF),
]

df_metrics = pd.DataFrame(resultados)

print("=" * 90)
print("  TABELA COMPARATIVA — QML vs Classicos")
print("  (mesmas features PCA, mesmo subconjunto de teste)")
print("=" * 90)
print(df_metrics.to_string(index=False))
print("=" * 90)

melhor = df_metrics.sort_values('AUC-ROC', ascending=False).iloc[0]
print(f"\n  Melhor AUC-ROC: {melhor['Modelo']} ({melhor['AUC-ROC']:.4f})")
print(f"  Melhor F1-Score: {df_metrics.sort_values('F1-Score',ascending=False).iloc[0]['Modelo']}")


In [ ]:
# ============================================================
# CÉLULA 13 — Curvas ROC Sobrepostas
# ============================================================
fig, ax = plt.subplots(figsize=(8, 6))

modelos_roc = [
    ('QSVC (QML)',           y_prob_qsvc, '#9C27B0', '--'),
    ('SVM-RBF (Classico)',   y_prob_svm,  '#2196F3', '-'),
    ('Random Forest (Class.)',y_prob_rf,  '#4CAF50', '-'),
]

for nome, probs, cor, ls in modelos_roc:
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc_val = roc_auc_score(y_test, probs)
    ax.plot(fpr, tpr, color=cor, linestyle=ls, linewidth=2.5,
            label=f'{nome} (AUC = {auc_val:.4f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, linewidth=1, label='Classificador Aleatorio')
ax.fill_between([0, 1], [0, 1], alpha=0.05, color='gray')
ax.set_xlabel('Taxa de Falsos Positivos (FPR)', fontsize=12)
ax.set_ylabel('Taxa de Verdadeiros Positivos (TPR)', fontsize=12)
ax.set_title('Curvas ROC Sobrepostas — QML vs Classicos\n'
             'Deteccao de Anomalias em Telemetria de Satelite',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Curvas ROC salvas em 'roc_curves.png'")


In [ ]:
# ============================================================
# CÉLULA 14 — Matrizes de Confusão Normalizadas
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Matrizes de Confusao Normalizadas\n'
             'Deteccao de Anomalias em Telemetria de Satelite',
             fontsize=13, fontweight='bold')

modelos_cm = [
    ('QSVC (QML)',             y_pred_qsvc, 'Purples'),
    ('SVM-RBF (Classico)',     y_pred_svm,  'Blues'),
    ('Random Forest (Class.)', y_pred_rf,   'Greens'),
]

for ax, (nome, preds, cmap) in zip(axes, modelos_cm):
    cm = confusion_matrix(y_test, preds, normalize='true')
    sns.heatmap(cm, annot=True, fmt='.3f', cmap=cmap, ax=ax,
                xticklabels=['Normal','Anomalia'],
                yticklabels=['Normal','Anomalia'],
                linewidths=0.5, vmin=0, vmax=1)
    ax.set_title(nome, fontweight='bold', fontsize=11)
    ax.set_xlabel('Predito')
    ax.set_ylabel('Real')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print("Matrizes de confusao salvas em 'confusion_matrices.png'")


In [ ]:
# ============================================================
# CÉLULA 15 — Gráfico Custo x Desempenho
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

modelos_nomes = [r['Modelo'].split(' ')[0] for r in resultados]
cores_bar = ['#9C27B0', '#2196F3', '#4CAF50']

# AUC-ROC
aucs = [r['AUC-ROC'] for r in resultados]
bars = axes[0].bar(modelos_nomes, aucs, color=cores_bar, edgecolor='black', alpha=0.85)
axes[0].set_title('AUC-ROC por Modelo', fontweight='bold')
axes[0].set_ylabel('AUC-ROC')
axes[0].set_ylim([max(0, min(aucs) - 0.1), 1.05])
axes[0].grid(axis='y', alpha=0.3)
for bar, val in zip(bars, aucs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.4f}', ha='center', va='bottom', fontweight='bold')

# Tempo treino
tempos = [r['T.Treino(s)'] for r in resultados]
bars2 = axes[1].bar(modelos_nomes, tempos, color=cores_bar, edgecolor='black', alpha=0.85)
axes[1].set_title('Tempo de Treinamento (s)', fontweight='bold')
axes[1].set_ylabel('Segundos')
axes[1].grid(axis='y', alpha=0.3)
for bar, val in zip(bars2, tempos):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.2f}s', ha='center', va='bottom', fontweight='bold')

plt.suptitle('Desempenho vs Custo Computacional', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('desempenho_custo.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 5. Análise dos Resultados e Conexão com o Problema Espacial

### Limitações NISQ identificadas
1. **Custo O(n²):** a kernel matrix do QSVC requer `n*(n-1)/2` avaliações de circuito → limita o treino quântico a ≤ 400 amostras
2. **Ruído de decoerência:** no hardware IBM Quantum real, erros de gate acumulam com a profundidade do circuito, degradando a fidelidade da kernel matrix
3. **Expressividade limitada:** ZZFeatureMap com reps=1 e entanglement linear captura apenas correlações de segunda ordem — problemas de alta complexidade exigiriam reps maiores

### Cenários favoráveis ao QML
- Dados com **estrutura geométrica intrínseca** no espaço de Hilbert (superposições quânticas capturam padrões não acessíveis ao kernel clássico)
- **Poucos dados de treino** (n < 1000): o custo quântico é competitivo com SVM clássico
- Features **altamente correlacionadas** com relações de segunda e terceira ordem (ZZ terms)

### Conexão com os ODS
- **ODS 9:** infraestrutura orbital mais confiável através de detecção precoce de anomalias
- **ODS 11:** alertas antecipados de falhas em satélites de monitoramento climático protegem comunidades costeiras
- **ODS 13:** satélites com maior vida útil reduzem descarte e lixo orbital, contribuindo para ação climática


In [ ]:
# ============================================================
# CÉLULA 16 — Análise Quantitativa Final
# ============================================================
print("=" * 65)
print("  ANALISE FINAL — QML vs Classicos")
print("=" * 65)

df_rank = df_metrics.sort_values('AUC-ROC', ascending=False).reset_index(drop=True)
qsvc_row = df_metrics[df_metrics['Modelo'].str.startswith('QSVC')].iloc[0]
classicos = df_metrics[~df_metrics['Modelo'].str.startswith('QSVC')]
best_class = classicos.sort_values('AUC-ROC', ascending=False).iloc[0]

delta_auc = qsvc_row['AUC-ROC'] - best_class['AUC-ROC']
delta_f1  = qsvc_row['F1-Score'] - best_class['F1-Score']
fator_tempo = qsvc_row['T.Treino(s)'] / best_class['T.Treino(s)']

print(f"\n  QSVC AUC-ROC : {qsvc_row['AUC-ROC']:.4f}")
print(f"  {best_class['Modelo']} AUC-ROC: {best_class['AUC-ROC']:.4f}")
print(f"  Delta AUC    : {delta_auc:+.4f}")
print(f"  Delta F1     : {delta_f1:+.4f}")
print(f"  Fator de tempo (QSVC / melhor classico): {fator_tempo:.1f}x")

if delta_auc >= 0:
    print(f"\n  QSVC supera ou iguala o melhor classico em AUC-ROC")
    print(f"  a um custo computacional {fator_tempo:.1f}x maior")
else:
    print(f"\n  Classico supera QSVC em AUC-ROC neste dataset")
    print(f"  (esperado em era NISQ: simuladores limitados a circuitos rasos)")

print("""
  Limitacoes NISQ:
  1. Custo O(n^2): kernel matrix limita treino a <=400 amostras
  2. Ruido de decoerencia degrada fidelidade em hardware real
  3. Circuito raso (reps=1) limita expressividade do feature map

  Impacto espacial:
  - Deteccao precoce de anomalias evita perda de missoes (> USD 200M/satelite)
  - QML pode detectar padroes anomalos em espaco de Hilbert nao acessiveis
    a kernels classicos para telemetria correlacionada multi-sensor
  - Com hardware quantico fault-tolerant (2030+), vantagem quantica esperada

  Conexao ODS:
  - ODS 9 : infraestrutura orbital confiavel
  - ODS 11: alertas antecipados para comunidades em risco
  - ODS 13: satelites com maior vida util reduzem lixo orbital
""")

print("=" * 65)
print("  Notebook executado com sucesso! (random_state=42)")
print("  Arquivos gerados:")
print("    eda_distribuicoes.png  | eda_correlacao.png")
print("    kernel_matrix.png      | roc_curves.png")
print("    confusion_matrices.png | desempenho_custo.png")
print("=" * 65)
